In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# carregar os dados de vendas do arquivo CSV

# df_dados_vendas = pd.read_csv(
#     r"C:\Users\tchie\OneDrive\Documentos\Tchiello\Python\Analise\codigos_py\Analise\Vendas_Hashtag\Vendas - Belo Horizonte.csv", 
#     low_memory=False, sep=",", encoding="utf-8")
# Para consolodar no mesmo data frame so dados de vendas de todas as cidades, basta usar o comando pd.concat() 
# Por exemplo:
df_arquivos_vendasbhrjsp = [
	"Vendas - Belo Horizonte.csv",
	"Vendas - Rio de Janeiro.csv",
	"Vendas - São Paulo.csv",
]

df_dados_vendas = pd.concat(
    [pd.read_csv(f"C:\\Users\\tchie\\OneDrive\\Documentos\\Tchiello\\Python\\Analise\\codigos_py\\Analise\\Vendas_Hashtag\\{arquivo}", 
                 low_memory=False, sep=",", encoding="utf-8") for arquivo in df_arquivos_vendasbhrjsp],
    ignore_index=True
)

In [15]:
# Inspeção de dados: verificar se há valores ausentes
# display(df_dados_vendas.head())
# display(df_dados_vendas.tail())
# display(df_dados_vendas.sample(5))
display(df_dados_vendas.shape)
display(list(df_dados_vendas.columns))
# Exibir as colunas como lista tem a vantagem de ser mais fácil de ler e manipular, 
# especialmente se você quiser acessar ou modificar as colunas posteriormente no código.
display(df_dados_vendas.info())
# através do df.info() é possível obter informações sobre o número de entradas, o número de colunas, 
# os tipos de dados de cada coluna e a quantidade de valores não nulos em cada coluna.
display(df_dados_vendas.describe())
# O objetivo é obter uma visão geral dos dados, identificar possíveis problemas, como valores ausentes ou tipos de dados incorretos, 
# e entender a distribuição dos dados para orientar as próximas etapas de análise.


(4342, 10)

['Unnamed: 0',
 'SKU',
 'Produto',
 'Quantidade Vendida',
 'Primeiro Nome',
 'Sobrenome',
 'Data',
 'Loja',
 'Preco Unitario',
 'Unnamed: 8']

<class 'pandas.DataFrame'>
RangeIndex: 4342 entries, 0 to 4341
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Unnamed: 0          4342 non-null   int64  
 1   SKU                 4342 non-null   str    
 2   Produto             4342 non-null   str    
 3   Quantidade Vendida  4342 non-null   int64  
 4   Primeiro Nome       4342 non-null   str    
 5   Sobrenome           4342 non-null   str    
 6   Data                4342 non-null   str    
 7   Loja                4342 non-null   str    
 8   Preco Unitario      4342 non-null   int64  
 9   Unnamed: 8          0 non-null      float64
dtypes: float64(1), int64(3), str(6)
memory usage: 339.3 KB


None

,Unnamed: 0,Quantidade Vendida,Preco Unitario,Unnamed: 8
count,4342.000000,4342.000000,4342.000000,0.0
mean,5011.788807,2.992170,3330.515891,NaN
std,2882.901663,1.392689,1456.810191,NaN
min,5.000000,1.000000,1400.000000,NaN
25%,2520.250000,2.000000,2100.000000,NaN
50%,5035.500000,3.000000,3400.000000,NaN
75%,7480.750000,4.000000,5300.000000,NaN
max,9996.000000,5.000000,5300.000000,NaN


In [21]:
# Tratamento de dados: renomear colunas e cálculo de Valor Total da Venda (com explicações)
# Possíveis erros ao executar a multiplicação:
# - KeyError: nomes de coluna diferentes (espaços, acentos, colunas não existentes)
# - TypeError/ValueError: colunas com valores não numéricos (strings, símbolos de moeda, separador de milhares)
# - Resultado com NaN: conversão falhou para algumas células (usar fillna ou remover)

# 1) Padronizar nomes de colunas (remove espaços acidentais no início/fim)
df_dados_vendas.columns = df_dados_vendas.columns.str.strip()

# 2) Inspecionar rapidamente as colunas e tipos antes de operar (didático)
print('Colunas disponíveis:', list(df_dados_vendas.columns))
display(df_dados_vendas[[col for col in ['Quantidade Vendida','Valor Unitário do Produto'] if col in df_dados_vendas.columns]].head())
print('\nDtypes:')
print(df_dados_vendas[[col for col in ['Quantidade Vendida','Valor Unitário do Produto'] if col in df_dados_vendas.columns]].dtypes)

# 3) Função utilitária para limpar e converter colunas numéricas com possíveis símbolos
def clean_numeric_series(s):
    # transforma para string, remove espaços e símbolos comuns de moeda, mantém dígitos, . , e -
    s = s.astype(str).str.strip()
    # remover símbolos de moeda e texto (ex: R$, USD, etc.) e letras
    s = s.str.replace(r'[^0-9,\.\-]', '', regex=True)
    # se o formato decimal usar vírgula, converte para ponto
    # este passo não quebra valores já no formato ponto
    s = s.str.replace(',', '.', regex=False)
    # converter para numérico (coerce converte erros para NaN)
    return pd.to_numeric(s, errors='coerce')

# 4) Aplicar limpeza às colunas necessárias (lança KeyError se coluna faltar)
required_cols = ['Quantidade Vendida','Valor Unitário do Produto']
for c in required_cols:
    if c not in df_dados_vendas.columns:
        raise KeyError(f'Coluna ausente: {c} - verifique nomes/encoding do seu CSV')
    df_dados_vendas[c] = clean_numeric_series(df_dados_vendas[c])

# 5) Diagnóstico: quantas conversões falharam? (ajuste de limpeza se necessário)
for c in required_cols:
    n_na = df_dados_vendas[c].isna().sum()
    print(f'Coluna {c}: {n_na} valores não convertidos (NaN) de', len(df_dados_vendas))

# 6) Opcional: lidar com NaN antes da multiplicação (exemplos comentados)
# - Preencher com 0 (pode alterar resultados): df_dados_vendas[required_cols] = df_dados_vendas[required_cols].fillna(0)
# - Remover linhas incompletas: df_dados_vendas.dropna(subset=required_cols, inplace=True)

# 7) Calcular o Valor Total da Venda de forma segura
df_dados_vendas['Valor Total da Venda'] = df_dados_vendas['Quantidade Vendida'] * df_dados_vendas['Valor Unitário do Produto']

# 8) Mostrar resultado e validar (didático)
display(df_dados_vendas[['Quantidade Vendida','Valor Unitário do Produto','Valor Total da Venda']].head())
print('\nResumo estatístico do Valor Total da Venda:')
display(df_dados_vendas['Valor Total da Venda'].describe())

# Tratamento de valores vazios
# se fosse excluir valores vazios, poderia usar o método dropna() do pandas, 
# mas isso pode resultar na perda de dados importantes.
# outro método é preencher os valores vazios com um valor específico, como a média ou a mediana da coluna,
# ou até mesmo com o método fillna() para preencher com o valor anterior ou posterior.
# Tratamento de datas
df_dados_vendas["Data da Venda"] = pd.to_datetime(df_dados_vendas["Data da Venda"], format="%d/%m/%Y")

# Para seaber otros formatos deve-se consultar a documentação do pandas sobre o parâmetro format do método to_datetime() 
# para entender como especificar corretamente o formato da data presente na coluna "Data da Venda".
# Tratamento de valores duplicados:
# Se houver valores duplicados, pode-se usar o método drop_duplicates() do pandas para removê-los.
# Exemplo: df_dados_vendas.drop_duplicates(inplace=True)
# Dentro desse processo de tratamento de dados, é importante considerar o contexto dos dados e os objetivos da análise 
# para tomar decisões informadas sobre como lidar com valores ausentes, formatos de data e valores duplicados.
# Um método e utilizar o subset() para selecionar apenas as colunas relevantes para a análise,
# e também usar o keep() para manter apenas as primeiras ou últimas ocorrências de valores duplicados,
# e depois aplicar o dropna() para remover as linhas com valores ausentes nessas colunas específicas.

Colunas disponíveis: ['Unnamed: 0', 'SKU', 'Produto', 'Quantidade Vendida', 'Primeiro Nome', 'Sobrenome', 'Data', 'Loja', 'Preco Unitario', 'Valor Total da Venda']


,Quantidade Vendida
0,2
1,1
2,5
3,5
4,5



Dtypes:
Quantidade Vendida    int64
dtype: object


KeyError: 'Coluna ausente: Valor Unitário do Produto - verifique nomes/encoding do seu CSV'